# Agentic RAG Platform — Local Demo Notebook

This notebook demonstrates all components of the `app/` Agentic RAG platform **locally**, without Docker or FastAPI.  
It mirrors the architecture of the deployed Azure-backed application but uses **local mocks / lightweight substitutes** so you can run everything on your machine.

## Sequence

1. **Pre-requisites & Environment Setup**
2. **Settings & Configuration**
3. **Local Client Initialisation** (replaces Azure Key Vault, Blob, PostgreSQL, Redis)
4. **Schemas / Data Models**
5. **RAG Service — Ingestion & Retrieval**
6. **Agent Tools** (Search, Postgres, Redis, Sandbox)
7. **SRE Agent**
8. **Engineering Agent**
9. **Streamlit Chat UI** (replaces FastAPI routes)

---

## Pre-requisites

Before running this notebook make sure you have the following:

### 1. Python ≥ 3.10

### 2. Install dependencies
```bash
pip install openai pydantic pydantic-settings langchain langchain-openai httpx streamlit
```

### 3. Azure OpenAI credentials
Create a `.env` file in the **project root** (`rag-infra/.env`) with at least:
```dotenv
AZURE_CLIENT_ID=<your-managed-identity-client-id-or-dummy>
AZURE_SEARCH_ENDPOINT=https://<your-search>.search.windows.net
AZURE_OPENAI_ENDPOINT=https://<your-openai>.openai.azure.com
AZURE_OPENAI_API_KEY=<your-api-key>
AZURE_OPENAI_DEPLOYMENT=gpt-4o
AZURE_OPENAI_EMBEDDING_DEPLOYMENT=text-embedding-3-large
BLOB_URI=https://<storageaccount>.blob.core.windows.net/raw-docs
KEYVAULT_URI=https://<vault>.vault.azure.net
POSTGRES_HOST=localhost
POSTGRES_DB=ragdb
POSTGRES_USER=postgres
REDIS_HOST=localhost
```
> **Tip:** For a fully local run you can leave the Azure values as placeholders — the notebook provides mock/fallback paths for every Azure service.

## 0. Imports & Path Setup

In [1]:
import sys, os, json, logging, hashlib, asyncio, traceback
from pathlib import Path
from typing import Any, Optional
from enum import Enum

# Ensure the project root is on sys.path so we can reference app/ structure
PROJECT_ROOT = Path.cwd().parent  # assumes notebook is in notebooks/
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Working directory:", os.getcwd())
print("Project root    :", PROJECT_ROOT)

Working directory: c:\Project\AI\rag-infra
Project root    : c:\Project\AI\rag-infra


## 1. Settings & Configuration

All configuration lives in `app/core/config.py`.  
It reads from a `.env` file via `pydantic-settings` and exposes Azure endpoints, model names, DB connection info, etc.

Below we recreate the `Settings` class so everything is self-contained in this notebook.

In [2]:
from pydantic_settings import BaseSettings, SettingsConfigDict


class Settings(BaseSettings):
    model_config = SettingsConfigDict(
        env_file=".env",
        env_file_encoding="utf-8",
        env_ignore_empty=True,
        extra="ignore",
    )

    # Azure Identity
    azure_client_id: str = ""

    # Azure AI Search
    azure_search_endpoint: str = ""
    azure_search_index_name: str = "agentic-rag-index"

    # Azure OpenAI
    azure_openai_endpoint: str = ""
    azure_openai_deployment: str = "gpt-4o"
    azure_openai_embedding_deployment: str = "text-embedding-3-large"
    azure_openai_api_version: str = "2024-05-01-preview"
    azure_openai_api_key: str = ""

    # Blob Storage
    blob_uri: str = ""

    # Key Vault
    keyvault_uri: str = ""
    kv_secret_pg_password: str = "Postgres-AdminPassword"
    kv_secret_redis_key: str = "Redis-PrimaryKey"
    kv_secret_openai_key: str = "AzureOpenAI-ApiKey"

    # PostgreSQL
    postgres_host: str = "localhost"
    postgres_db: str = "ragdb"
    postgres_user: str = "postgres"

    # Redis
    redis_host: str = "localhost"
    redis_ssl_port: int = 6380

    # App Insights (optional)
    applicationinsights_connection_string: str = ""

    # Entra ID
    entra_tenant_id: str = ""
    entra_audience: str = ""


settings = Settings()

print(f"Azure OpenAI endpoint   : {settings.azure_openai_endpoint or '(not set)'}")
print(f"Azure OpenAI deployment : {settings.azure_openai_deployment}")
print(f"Azure OpenAI API key    : {'***' + settings.azure_openai_api_key[-4:] if settings.azure_openai_api_key else '(not set)'}")
print(f"Search endpoint         : {settings.azure_search_endpoint or '(not set)'}")
print(f"Postgres host           : {settings.postgres_host}")
print(f"Redis host              : {settings.redis_host}")

Azure OpenAI endpoint   : (not set)
Azure OpenAI deployment : gpt-4o
Azure OpenAI API key    : (not set)
Search endpoint         : (not set)
Postgres host           : localhost
Redis host              : localhost


## 2. Schemas / Data Models

These are the Pydantic models from `app/models/schemas.py`.  
They define the request/response shapes used by the agents and the UI.

In [3]:
from pydantic import BaseModel, Field


class AgentType(str, Enum):
    sre = "sre"
    engineering = "engineering"


class ChatRequest(BaseModel):
    agent: AgentType = AgentType.sre
    session_id: str = Field(..., description="Unique session/conversation ID")
    message: str = Field(..., min_length=1, max_length=4096)


class ChatResponse(BaseModel):
    session_id: str
    agent: AgentType
    answer: str
    sources: list[str] = []
    tool_calls: list[str] = []


class IngestRequest(BaseModel):
    container: str = "raw-docs"
    blob_prefix: str = ""
    force_reindex: bool = False


class IngestResponse(BaseModel):
    status: str
    documents_indexed: int
    errors: list[str] = []


class DocumentItem(BaseModel):
    name: str
    size: int
    last_modified: str
    uri: str


class DocumentsResponse(BaseModel):
    container: str
    documents: list[DocumentItem]


class ServiceStatus(str, Enum):
    ok = "ok"
    degraded = "degraded"
    error = "error"


class HealthResponse(BaseModel):
    status: ServiceStatus
    services: dict[str, Any]


# Quick validation
sample = ChatRequest(agent="sre", session_id="demo-001", message="Why is the API latency high?")
print("Sample ChatRequest:", sample.model_dump_json(indent=2))

Sample ChatRequest: {
  "agent": "sre",
  "session_id": "demo-001",
  "message": "Why is the API latency high?"
}


## 3. Local Client Initialisation

In production the app uses Azure Managed Identity to initialise Key Vault, Blob Storage, AI Search, OpenAI, PostgreSQL, and Redis clients (see `app/core/clients.py`).

For this **local demo** we create:
- An **Azure OpenAI** client using an API key (if provided in `.env`)
- A **local in-memory Redis substitute** (dict-based)
- A **local SQLite substitute** for PostgreSQL
- A **local file-based ingestion** substitute for Blob Storage

> If you have a real Azure OpenAI endpoint + key, the LLM calls will work end-to-end.  
> Otherwise the agent cells will show fallback/mock responses.

In [ ]:
import sqlite3
from openai import AzureOpenAI  # synchronous client for notebook use

logger = logging.getLogger("demo")
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")


# ── 3a. Azure OpenAI client ──────────────────────────────────────────────────
if not settings.azure_openai_api_key:
    raise ValueError(
        "AZURE_OPENAI_API_KEY is not set. "
        "Please add it to your .env file in the project root."
    )
if not settings.azure_openai_endpoint:
    raise ValueError(
        "AZURE_OPENAI_ENDPOINT is not set. "
        "Please add it to your .env file in the project root."
    )

openai_client = AzureOpenAI(
    azure_endpoint=settings.azure_openai_endpoint,
    api_key=settings.azure_openai_api_key,
    api_version=settings.azure_openai_api_version,
)
print("✅ Azure OpenAI client initialised (API-key auth)")


# ── 3b. Local Redis substitute (in-memory dict) ─────────────────────────────
class LocalRedis:
    """Drop-in mock for redis — stores conversation history in a dict."""

    def __init__(self):
        self._store: dict[str, str] = {}

    def get(self, key: str) -> Optional[str]:
        return self._store.get(key)

    def setex(self, key: str, ttl: int, value: str):
        self._store[key] = value  # TTL ignored locally

    def ping(self) -> bool:
        return True


redis_client = LocalRedis()
print("✅ Local Redis (in-memory dict) ready")


# ── 3c. Local SQLite substitute for PostgreSQL ──────────────────────────────
DB_PATH = PROJECT_ROOT / "data" / "demo_local.db"
DB_PATH.parent.mkdir(parents=True, exist_ok=True)


def _get_db() -> sqlite3.Connection:
    conn = sqlite3.connect(str(DB_PATH), check_same_thread=False)
    conn.row_factory = sqlite3.Row
    return conn


def _init_local_pg():
    """Create tables that mirror the production PostgreSQL schema."""
    with _get_db() as conn:
        conn.executescript("""
            CREATE TABLE IF NOT EXISTS incidents (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                service TEXT NOT NULL,
                severity TEXT NOT NULL DEFAULT 'medium',
                title TEXT NOT NULL,
                started_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                resolved_at TIMESTAMP,
                summary TEXT
            );
            CREATE TABLE IF NOT EXISTS service_dependencies (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                upstream TEXT NOT NULL,
                downstream TEXT NOT NULL,
                dependency_type TEXT NOT NULL DEFAULT 'http'
            );
            CREATE TABLE IF NOT EXISTS agent_interactions (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                session_id TEXT NOT NULL,
                agent TEXT NOT NULL,
                query TEXT NOT NULL,
                answer TEXT NOT NULL,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            );
        """)


_init_local_pg()
print(f"✅ Local SQLite DB initialised at {DB_PATH}")


# ── 3d. Local document directory (replaces Azure Blob Storage) ───────────────
LOCAL_DOCS_DIR = PROJECT_ROOT / "data" / "raw-docs"
LOCAL_DOCS_DIR.mkdir(parents=True, exist_ok=True)
print(f"✅ Local docs directory: {LOCAL_DOCS_DIR}")
print("   Place .md / .txt files here for ingestion")

⚠️  Azure OpenAI credentials not set — LLM calls will use mock responses
✅ Local Redis (in-memory dict) ready
✅ Local SQLite DB initialised at c:\Project\AI\rag-infra\data\demo_local.db
✅ Local docs directory: c:\Project\AI\rag-infra\data\raw-docs
   Place .md / .txt files here for ingestion


### 3e. Seed sample data

Insert a few sample incidents and service dependencies so the agent tools have something to query.

In [5]:
with _get_db() as conn:
    # Seed incidents (idempotent — skip if rows exist)
    existing = conn.execute("SELECT COUNT(*) FROM incidents").fetchone()[0]
    if existing == 0:
        conn.executemany(
            "INSERT INTO incidents (service, severity, title, summary) VALUES (?, ?, ?, ?)",
            [
                ("payment-service", "high", "Payment gateway 5xx spike",
                 "Intermittent 502s from Stripe webhook handler; root cause was connection pool exhaustion."),
                ("auth-service", "critical", "OAuth token endpoint down",
                 "Expired TLS cert on auth-service caused 100% failures for 12 minutes."),
                ("order-service", "medium", "Slow order confirmation emails",
                 "SQS queue lag reached 45 s due to under-provisioned consumers."),
                ("payment-service", "low", "Minor logging noise in payment-service",
                 "Debug-level logs flooding CloudWatch; log level bumped to INFO."),
            ],
        )
        print("Seeded 4 sample incidents")
    else:
        print(f"Incidents table already has {existing} rows — skipping seed")

    # Seed service dependencies
    existing_deps = conn.execute("SELECT COUNT(*) FROM service_dependencies").fetchone()[0]
    if existing_deps == 0:
        conn.executemany(
            "INSERT INTO service_dependencies (upstream, downstream, dependency_type) VALUES (?, ?, ?)",
            [
                ("api-gateway", "auth-service", "http"),
                ("api-gateway", "order-service", "http"),
                ("order-service", "payment-service", "http"),
                ("order-service", "inventory-service", "grpc"),
                ("payment-service", "stripe-webhook", "http"),
                ("notification-service", "order-service", "async/sqs"),
            ],
        )
        print("Seeded 6 sample service dependencies")
    else:
        print(f"Dependencies table already has {existing_deps} rows — skipping seed")

Seeded 4 sample incidents
Seeded 6 sample service dependencies


## 4. Agent Tools

The platform exposes four tool modules that agents can invoke:

| Tool | Production backend | Local substitute |
|---|---|---|
| `search_tool` — hybrid search + embedding | Azure AI Search + Azure OpenAI Embeddings | Simple TF-IDF keyword search over local docs |
| `postgres_tool` — incident history, service deps, audit log | Azure PostgreSQL | Local SQLite |
| `redis_tool` — conversation history & cache | Azure Cache for Redis | In-memory dict |
| `sandbox_tool` — restricted code execution | Same (pure Python) | Same (pure Python) |

### 4a. Search Tool (local version)

In production this uses Azure AI Search with hybrid (keyword + vector) search.  
Locally we use a simple keyword-match over ingested document chunks.

In [6]:
# ── Local search index (replaces Azure AI Search) ────────────────────────────
# Stores document chunks in memory; uses simple keyword overlap scoring.

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100

_local_search_index: list[dict[str, Any]] = []  # in-memory document store


def _chunk_text(text: str) -> list[str]:
    """Split text into overlapping chunks (mirrors app/services/ingestion_service.py)."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + CHUNK_SIZE, len(text))
        chunks.append(text[start:end])
        start += CHUNK_SIZE - CHUNK_OVERLAP
    return chunks


def get_embedding(text: str) -> list[float]:
    """
    Get embedding vector from Azure OpenAI.
    Falls back to an empty list when the client is unavailable.
    """
    if openai_client:
        try:
            response = openai_client.embeddings.create(
                input=text,
                model=settings.azure_openai_embedding_deployment,
            )
            return response.data[0].embedding
        except Exception as exc:
            logger.warning("Embedding call failed: %s — returning empty vector", exc)
    return []


def local_ingest_directory(directory: Path) -> dict[str, int]:
    """Index .md and .txt files from a local directory into the in-memory search index."""
    global _local_search_index
    source_files = sorted([*directory.glob("*.md"), *directory.glob("*.txt")])
    count = 0
    for file_path in source_files:
        text = file_path.read_text(encoding="utf-8", errors="ignore")
        chunks = _chunk_text(text)
        for i, chunk in enumerate(chunks):
            doc_id = hashlib.md5(f"{file_path.name}:{i}".encode()).hexdigest()
            _local_search_index.append({
                "id": doc_id,
                "content": chunk,
                "title": file_path.name,
                "source": str(file_path),
            })
            count += 1
    return {"files_indexed": len(source_files), "chunks_indexed": count}


def hybrid_search(query: str, top_k: int = 5) -> list[dict[str, Any]]:
    """
    Local keyword-based search over the in-memory index.
    Production uses Azure AI Search hybrid (keyword + vector) search.
    """
    query_words = set(query.lower().split())
    scored = []
    for doc in _local_search_index:
        content_lower = doc["content"].lower()
        score = sum(1 for w in query_words if w in content_lower)
        if score > 0:
            scored.append({**doc, "score": score})
    scored.sort(key=lambda x: x["score"], reverse=True)
    return scored[:top_k]


# Test: ingest any docs already in the local docs directory
ingest_stats = local_ingest_directory(LOCAL_DOCS_DIR)
print(f"Local search index: {ingest_stats}")
print(f"Total chunks in index: {len(_local_search_index)}")

Local search index: {'files_indexed': 0, 'chunks_indexed': 0}
Total chunks in index: 0


### 4b. Postgres Tool (local SQLite version)

Mirrors `app/agents/tools/postgres_tool.py` — incident history, service dependencies, and audit logging.

In [7]:
def query_incident_history(service_name: str, limit: int = 10) -> list[dict]:
    """Query recent incidents for a given service (local SQLite version)."""
    with _get_db() as conn:
        rows = conn.execute(
            "SELECT id, service, severity, title, started_at, resolved_at, summary "
            "FROM incidents WHERE service = ? ORDER BY started_at DESC LIMIT ?",
            (service_name, limit),
        ).fetchall()
    return [dict(r) for r in rows]


def query_service_dependencies(service_name: str) -> list[dict]:
    """Query service dependency graph (local SQLite version)."""
    with _get_db() as conn:
        rows = conn.execute(
            "SELECT upstream, downstream, dependency_type "
            "FROM service_dependencies WHERE upstream = ? OR downstream = ?",
            (service_name, service_name),
        ).fetchall()
    return [dict(r) for r in rows]


def log_agent_interaction(session_id: str, agent: str, query: str, answer: str):
    """Persist agent interaction for audit (local SQLite version)."""
    with _get_db() as conn:
        conn.execute(
            "INSERT INTO agent_interactions (session_id, agent, query, answer) VALUES (?, ?, ?, ?)",
            (session_id, agent, query, answer),
        )


# Test the tools
print("Incidents for 'payment-service':")
for inc in query_incident_history("payment-service"):
    print(f"  [{inc['severity']}] {inc['title']}")

print("\nDependencies involving 'order-service':")
for dep in query_service_dependencies("order-service"):
    print(f"  {dep['upstream']} → {dep['downstream']} ({dep['dependency_type']})")

Incidents for 'payment-service':
  [high] Payment gateway 5xx spike
  [low] Minor logging noise in payment-service

Dependencies involving 'order-service':
  api-gateway → order-service (http)
  order-service → payment-service (http)
  order-service → inventory-service (grpc)
  notification-service → order-service (async/sqs)


### 4c. Redis Tool (local in-memory version)

Mirrors `app/agents/tools/redis_tool.py` — conversation history and caching.

In [8]:
HISTORY_TTL = 3600  # 1 hour (ignored locally, kept for parity)


def get_conversation_history(session_id: str) -> list[dict]:
    """Retrieve conversation history from local Redis substitute."""
    raw = redis_client.get(f"session:{session_id}:history")
    if not raw:
        return []
    return json.loads(raw)


def append_to_history(session_id: str, role: str, content: str):
    """Append a message to the conversation history (keeps last 20 turns)."""
    history = get_conversation_history(session_id)
    history.append({"role": role, "content": content})
    history = history[-20:]
    redis_client.setex(
        f"session:{session_id}:history",
        HISTORY_TTL,
        json.dumps(history),
    )


def cache_set(key: str, value: str, ttl: int = 300):
    redis_client.setex(key, ttl, value)


def cache_get(key: str) -> Optional[str]:
    return redis_client.get(key)


# Test
append_to_history("demo-001", "user", "Hello, what is the payment-service?")
append_to_history("demo-001", "assistant", "The payment-service handles all payment processing.")
history = get_conversation_history("demo-001")
print(f"Conversation history ({len(history)} messages):")
for msg in history:
    print(f"  [{msg['role']}] {msg['content'][:80]}")

Conversation history (2 messages):
  [user] Hello, what is the payment-service?
  [assistant] The payment-service handles all payment processing.


### 4d. Sandbox Tool

Executes untrusted Python code in a restricted sandbox.  
This is the same implementation as `app/agents/tools/sandbox_tool.py` — no Azure dependency.

In [9]:
import builtins as _builtins_mod

SANDBOX_TIMEOUT = 10  # seconds
ALLOWED_BUILTINS = {
    "print", "len", "range", "enumerate", "zip",
    "list", "dict", "set", "tuple", "str", "int",
    "float", "bool", "type", "isinstance", "min", "max", "sum",
}


def execute_code(code: str) -> dict[str, Any]:
    """
    Execute untrusted Python code in a restricted sandbox.
    No file I/O, no imports, no network — pure computation only.
    (Synchronous version for notebook use; production uses async.)
    """
    restricted_globals = {
        "__builtins__": {k: getattr(_builtins_mod, k) for k in ALLOWED_BUILTINS if hasattr(_builtins_mod, k)},
    }
    output_lines: list[str] = []

    def _capture_print(*args, **kwargs):
        output_lines.append(" ".join(str(a) for a in args))

    restricted_globals["__builtins__"]["print"] = _capture_print

    try:
        local_vars: dict = {}
        exec(compile(code, "<sandbox>", "exec"), restricted_globals, local_vars)
        return {
            "status": "ok",
            "output": "\n".join(output_lines),
            "locals": {k: repr(v) for k, v in local_vars.items()},
        }
    except Exception:
        return {"status": "error", "output": traceback.format_exc(limit=5)}


# Test
result = execute_code("x = sum(range(10))\nprint('Sum:', x)")
print("Sandbox result:", json.dumps(result, indent=2))

Sandbox result: {
  "status": "ok",
  "output": "Sum: 45",
  "locals": {
    "x": "45"
  }
}


## 5. RAG Service — Ingestion & Retrieval

`retrieve_context()` mirrors `app/services/rag_service.py`.  
It searches the local index and builds a context string for the LLM.

`ingest_local_docs()` mirrors `app/services/ingestion_service.py`.  
It reads files from `data/raw-docs/` instead of Azure Blob Storage.

In [10]:
CONTEXT_MAX_CHARS = 6000


def retrieve_context(query: str, top_k: int = 5) -> tuple[str, list[str]]:
    """
    Retrieve relevant chunks and build a context string.
    Mirrors app/services/rag_service.py but uses the local search index.
    """
    docs = hybrid_search(query, top_k=top_k)

    context_parts = []
    sources = []
    total_chars = 0

    for doc in docs:
        chunk = f"[{doc['title']}]\n{doc['content']}"
        if total_chars + len(chunk) > CONTEXT_MAX_CHARS:
            break
        context_parts.append(chunk)
        sources.append(doc["source"])
        total_chars += len(chunk)

    context = "\n\n---\n\n".join(context_parts)
    return context, list(set(sources))


def ingest_local_docs(directory: Path | None = None) -> dict[str, int]:
    """
    Ingest .md and .txt files from a local directory.
    Mirrors app/services/ingestion_service.py (which reads from Azure Blob Storage).
    """
    target = directory or LOCAL_DOCS_DIR
    return local_ingest_directory(target)


# Create a sample document for testing if the docs directory is empty
sample_doc = LOCAL_DOCS_DIR / "runbook_payment_service.md"
if not sample_doc.exists():
    sample_doc.write_text(
        "# Payment Service Runbook\n\n"
        "## Overview\n"
        "The payment-service processes all payment transactions via Stripe.\n\n"
        "## Common Issues\n"
        "### 5xx Errors\n"
        "- Check connection pool settings in `config/pool.yaml`.\n"
        "- Verify Stripe API key is valid and not rate-limited.\n"
        "- Inspect CloudWatch logs for `PoolExhausted` exceptions.\n\n"
        "### High Latency\n"
        "- Check database connection pool utilisation.\n"
        "- Review recent deployments for regression.\n"
        "- Verify downstream Stripe endpoint health at https://status.stripe.com.\n\n"
        "## Escalation\n"
        "If unresolved within 15 minutes, page the payments-oncall rotation.\n",
        encoding="utf-8",
    )
    print("Created sample runbook:", sample_doc.name)

# Re-ingest with the new sample doc
ingest_result = ingest_local_docs()
print("Ingest result:", json.dumps(ingest_result, indent=2))

# Test retrieval
context, sources = retrieve_context("payment service 5xx errors")
print(f"\nRetrieved context ({len(context)} chars) from {len(sources)} source(s)")
if context:
    print(context[:500])

Created sample runbook: runbook_payment_service.md
Ingest result: {
  "files_indexed": 1,
  "chunks_indexed": 1
}

Retrieved context (601 chars) from 1 source(s)
[runbook_payment_service.md]
# Payment Service Runbook

## Overview
The payment-service processes all payment transactions via Stripe.

## Common Issues
### 5xx Errors
- Check connection pool settings in `config/pool.yaml`.
- Verify Stripe API key is valid and not rate-limited.
- Inspect CloudWatch logs for `PoolExhausted` exceptions.

### High Latency
- Check database connection pool utilisation.
- Review recent deployments for regression.
- Verify downstream Stripe endpoint health at https://s


## 6. SRE Agent

The SRE agent (`app/agents/sre_agent.py`) analyses incidents, suggests root cause analysis, and helps with on-call triage.

It combines:
1. Conversation history (Redis)
2. RAG context (AI Search)
3. Incident history (PostgreSQL)
4. Sandbox execution (if code block in message)
5. Azure OpenAI LLM call

In [ ]:
SRE_SYSTEM_PROMPT = """
You are an expert SRE (Site Reliability Engineer) AI assistant.
Your responsibilities:
- Analyze incidents and alerts based on historical data and runbooks.
- Suggest root cause analysis (RCA) and remediation steps.
- Help with on-call triage, runbook lookup, and postmortem drafting.
- Answer questions about service dependencies and SLOs/SLIs.
- Execute diagnostic code snippets safely when needed.

Always:
- Ground your answers in retrieved context from the knowledge base.
- Cite sources when referencing runbooks or past incidents.
- Be concise, structured (use numbered steps for procedures).
- Never reveal secrets, connection strings, or internal credentials.
"""

# Common English words to skip during service name detection
_STOP_WORDS = {
    "what", "when", "where", "which", "there", "their", "about",
    "would", "could", "should", "have", "been", "that", "this",
    "with", "from", "your", "into", "will", "more", "also",
}


def run_sre_agent(session_id: str, user_message: str) -> dict[str, Any]:
    """
    Local synchronous version of app/agents/sre_agent.py.
    Uses local tools instead of Azure services.
    Calls Azure OpenAI for LLM responses.
    """
    logger.info("SRE agent: session=%s query='%s'", session_id, user_message[:80])

    # 1) Conversation history from local Redis
    history = []
    try:
        history = get_conversation_history(f"sre:{session_id}")
    except Exception as exc:
        logger.warning("History fetch failed: %s", exc)

    # 2) RAG context from local search index
    context, sources = retrieve_context(user_message, top_k=5)

    # 3) Incident lookup from local SQLite
    incidents = []
    tool_calls = []
    words = user_message.lower().split()
    for word in words:
        if len(word) > 4 and word not in _STOP_WORDS:
            rows = query_incident_history(word)
            if rows:
                incidents = rows
                tool_calls.append(f"query_incident_history(service={word})")
                break

    # 4) Sandbox execution (if code block detected)
    sandbox_result = None
    if "```python" in user_message:
        start = user_message.find("```python") + 9
        end = user_message.find("```", start)
        if end > start:
            code_block = user_message[start:end].strip()
            sandbox_result = execute_code(code_block)
            tool_calls.append("execute_code(sandbox)")

    # 5) Build messages
    messages = [{"role": "system", "content": SRE_SYSTEM_PROMPT}]

    if context:
        messages.append({"role": "system", "content": f"Relevant knowledge base context:\n\n{context}"})

    if incidents:
        incident_text = "\n".join(
            f"- [{r['severity']}] {r['title']} at {r['started_at']}: {r.get('summary', '')}"
            for r in incidents
        )
        messages.append({"role": "system", "content": f"Recent incidents:\n{incident_text}"})

    if sandbox_result:
        messages.append({
            "role": "system",
            "content": f"Sandbox execution result:\nStatus: {sandbox_result['status']}\nOutput:\n{sandbox_result['output']}",
        })

    messages.extend(history[-10:])
    messages.append({"role": "user", "content": user_message})

    # 6) Call Azure OpenAI
    try:
        response = openai_client.chat.completions.create(
            model=settings.azure_openai_deployment,
            messages=messages,
            temperature=0.2,
            max_tokens=1500,
        )
        answer = response.choices[0].message.content
    except Exception as exc:
        answer = f"[LLM call failed: {exc}]\n\nContext retrieved:\n{context[:500] if context else 'None'}"

    # 7) Persist to local Redis + SQLite
    try:
        append_to_history(f"sre:{session_id}", "user", user_message)
        append_to_history(f"sre:{session_id}", "assistant", answer)
    except Exception as exc:
        logger.warning("History save failed: %s", exc)

    try:
        log_agent_interaction(session_id, "sre", user_message, answer)
    except Exception as exc:
        logger.warning("Interaction log failed: %s", exc)

    return {"answer": answer, "sources": sources, "tool_calls": tool_calls}


print("✅ SRE agent defined")

✅ SRE agent defined


### 6a. Test the SRE Agent

In [12]:
sre_result = run_sre_agent(
    session_id="demo-sre-001",
    user_message="We're seeing 5xx errors on payment-service. What should I check first?",
)

print("=" * 60)
print("SRE AGENT RESPONSE")
print("=" * 60)
print(sre_result["answer"])
print(f"\nSources: {sre_result['sources']}")
print(f"Tool calls: {sre_result['tool_calls']}")

INFO | SRE agent: session=demo-sre-001 query='We're seeing 5xx errors on payment-service. What should I check first?'


SRE AGENT RESPONSE
[Mock SRE Response — Azure OpenAI not configured]

Based on the retrieved context (1 source(s)) and 0 incident record(s), here is a summary:


Relevant documentation:
[runbook_payment_service.md]
# Payment Service Runbook

## Overview
The payment-service processes all payment transactions via Stripe.

## Common Issues
### 5xx Errors
- Check connection pool settings in `config/pool.yaml`.
- Verify Stripe API key is valid and not rate-limited.
- Inspect CloudWatch logs for `PoolExhausted` exceptions.

### High Latency
- Check database connection pool utilisation.

Sources: ['c:\\Project\\AI\\rag-infra\\data\\raw-docs\\runbook_payment_service.md']
Tool calls: []


## 7. Engineering Agent

The Engineering agent (`app/agents/engineering_agent.py`) answers architecture, design, and code-related questions.

It combines:
1. Conversation history (Redis)
2. RAG context (AI Search)
3. Service dependency lookup (PostgreSQL)
4. Sandbox execution (if code block in message)
5. Azure OpenAI LLM call

In [ ]:
ENGINEERING_SYSTEM_PROMPT = """
You are an expert Software Engineering AI assistant embedded in an engineering platform.
Your responsibilities:
- Answer architecture, design, and code-related questions.
- Review code snippets and suggest improvements.
- Explain service dependencies and integration patterns.
- Help with debugging, performance analysis, and best practices.
- Execute safe code snippets in a sandboxed environment.

Always:
- Ground answers in retrieved internal documentation.
- Cite sources (ADRs, RFCs, wiki pages) when relevant.
- Be precise and use concrete examples.
- Prefer idiomatic, production-ready code suggestions.
- Never output secrets, credentials, or connection strings.
"""


def run_engineering_agent(session_id: str, user_message: str) -> dict[str, Any]:
    """
    Local synchronous version of app/agents/engineering_agent.py.
    Uses local tools instead of Azure services.
    Calls Azure OpenAI for LLM responses.
    """
    logger.info("Engineering agent: session=%s query='%s'", session_id, user_message[:80])

    # 1) Conversation history
    history = []
    try:
        history = get_conversation_history(f"eng:{session_id}")
    except Exception as exc:
        logger.warning("History fetch failed: %s", exc)

    # 2) RAG context
    context, sources = retrieve_context(user_message, top_k=5)

    # 3) Service dependency lookup
    dependencies = []
    tool_calls = []
    words = user_message.lower().split()
    for word in words:
        if len(word) > 4 and word not in _STOP_WORDS:
            rows = query_service_dependencies(word)
            if rows:
                dependencies = rows
                tool_calls.append(f"query_service_dependencies(service={word})")
                break

    # 4) Sandbox execution (if code block detected)
    sandbox_result = None
    if "```python" in user_message:
        start = user_message.find("```python") + 9
        end = user_message.find("```", start)
        if end > start:
            code_block = user_message[start:end].strip()
            sandbox_result = execute_code(code_block)
            tool_calls.append("execute_code(sandbox)")

    # 5) Build messages
    messages = [{"role": "system", "content": ENGINEERING_SYSTEM_PROMPT}]

    if context:
        messages.append({"role": "system", "content": f"Relevant internal documentation:\n\n{context}"})

    if dependencies:
        dep_text = "\n".join(
            f"- {r['upstream']} → {r['downstream']} ({r['dependency_type']})"
            for r in dependencies
        )
        messages.append({"role": "system", "content": f"Service dependencies:\n{dep_text}"})

    if sandbox_result:
        messages.append({
            "role": "system",
            "content": f"Sandbox execution result:\nStatus: {sandbox_result['status']}\nOutput:\n{sandbox_result['output']}",
        })

    messages.extend(history[-10:])
    messages.append({"role": "user", "content": user_message})

    # 6) Call Azure OpenAI
    try:
        response = openai_client.chat.completions.create(
            model=settings.azure_openai_deployment,
            messages=messages,
            temperature=0.3,
            max_tokens=2000,
        )
        answer = response.choices[0].message.content
    except Exception as exc:
        answer = f"[LLM call failed: {exc}]\n\nContext retrieved:\n{context[:500] if context else 'None'}"

    # 7) Persist history + audit log
    try:
        append_to_history(f"eng:{session_id}", "user", user_message)
        append_to_history(f"eng:{session_id}", "assistant", answer)
    except Exception as exc:
        logger.warning("History save failed: %s", exc)

    try:
        log_agent_interaction(session_id, "engineering", user_message, answer)
    except Exception as exc:
        logger.warning("Interaction log failed: %s", exc)

    return {"answer": answer, "sources": sources, "tool_calls": tool_calls}


print("✅ Engineering agent defined")

✅ Engineering agent defined


### 7a. Test the Engineering Agent

In [14]:
eng_result = run_engineering_agent(
    session_id="demo-eng-001",
    user_message="What are the dependencies of order-service and how does it connect to payment-service?",
)

print("=" * 60)
print("ENGINEERING AGENT RESPONSE")
print("=" * 60)
print(eng_result["answer"])
print(f"\nSources: {eng_result['sources']}")
print(f"Tool calls: {eng_result['tool_calls']}")

INFO | Engineering agent: session=demo-eng-001 query='What are the dependencies of order-service and how does it connect to payment-se'


ENGINEERING AGENT RESPONSE
[Mock Engineering Response — Azure OpenAI not configured]

Based on the retrieved context (1 source(s)) and 4 dependency record(s), here is a summary:

• api-gateway → order-service (http)
• order-service → payment-service (http)
• order-service → inventory-service (grpc)
• notification-service → order-service (async/sqs)

Relevant documentation:
[runbook_payment_service.md]
# Payment Service Runbook

## Overview
The payment-service processes all payment transactions via Stripe.

## Common Issues
### 5xx Errors
- Check connection pool settings in `config/pool.yaml`.
- Verify Stripe API key is valid and not rate-limited.
- Inspect CloudWatch logs for `PoolExhausted` exceptions.

### High Latency
- Check database connection pool utilisation.

Sources: ['c:\\Project\\AI\\rag-infra\\data\\raw-docs\\runbook_payment_service.md']
Tool calls: ['query_service_dependencies(service=order-service)']


## 8. Health Check

Mirrors `app/api/routes/health.py` — verifies connectivity to all backend services.

In [ ]:
def check_health() -> dict[str, Any]:
    """
    Local health check — mirrors app/api/routes/health.py.
    Checks local Redis, SQLite, OpenAI client, and search index.
    """
    services = {}
    overall = "ok"

    # Redis
    try:
        redis_client.ping()
        services["redis"] = "ok (local in-memory)"
    except Exception as exc:
        services["redis"] = f"error: {exc}"
        overall = "degraded"

    # PostgreSQL (SQLite)
    try:
        with _get_db() as conn:
            conn.execute("SELECT 1").fetchone()
        services["postgres"] = "ok (local SQLite)"
    except Exception as exc:
        services["postgres"] = f"error: {exc}"
        overall = "degraded"

    # Azure OpenAI — verify with a lightweight API call
    try:
        openai_client.chat.completions.create(
            model=settings.azure_openai_deployment,
            messages=[{"role": "user", "content": "ping"}],
            max_tokens=5,
        )
        services["azure_openai"] = "ok (API key auth)"
    except Exception as exc:
        services["azure_openai"] = f"degraded: {exc}"
        overall = "degraded"

    # Search index
    services["ai_search"] = f"ok (local — {len(_local_search_index)} chunks indexed)"

    return {"status": overall, "services": services}


health = check_health()
print("Health Check:")
print(json.dumps(health, indent=2))

Health Check:
{
  "status": "degraded",
  "services": {
    "redis": "ok (local in-memory)",
    "postgres": "ok (local SQLite)",
    "azure_openai": "not configured (mock mode)",
    "ai_search": "ok (local \u2014 1 chunks indexed)"
  }
}


## 9. Streamlit Chat UI

Instead of FastAPI routes (`app/api/routes/chat.py`, `ingest.py`, `documents.py`, `health.py`), we provide a **Streamlit** app that exposes the same functionality through a browser UI.

The cell below writes a `streamlit_app.py` file to disk and provides instructions to run it.

### Features
- **Agent selector** — choose between SRE and Engineering agents
- **Chat interface** — multi-turn conversation with history
- **Document ingestion** — upload `.md` / `.txt` files into the local search index
- **Health dashboard** — shows status of all local services

In [17]:
streamlit_app_code = r'''
"""
Streamlit Chat UI for Agentic RAG Platform
==========================================
Replaces the FastAPI routes with a simple browser-based interface.

Run with:
    streamlit run streamlit_app.py
"""
import streamlit as st
import sys, os, json, hashlib, logging, sqlite3, traceback, uuid
from pathlib import Path
from typing import Any, Optional
from enum import Enum

# ── Ensure project root is importable ─────────────────────────────────────────
PROJECT_ROOT = Path(__file__).resolve().parent.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pydantic_settings import BaseSettings, SettingsConfigDict

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("streamlit_app")

# ══════════════════════════════════════════════════════════════════════════════
# Settings (from app/core/config.py)
# ══════════════════════════════════════════════════════════════════════════════
class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_file=".env", env_file_encoding="utf-8", env_ignore_empty=True, extra="ignore")
    azure_client_id: str = ""
    azure_search_endpoint: str = ""
    azure_search_index_name: str = "agentic-rag-index"
    azure_openai_endpoint: str = ""
    azure_openai_deployment: str = "gpt-4o"
    azure_openai_embedding_deployment: str = "text-embedding-3-large"
    azure_openai_api_version: str = "2024-05-01-preview"
    azure_openai_api_key: str = ""
    blob_uri: str = ""
    keyvault_uri: str = ""
    postgres_host: str = "localhost"
    postgres_db: str = "ragdb"
    postgres_user: str = "postgres"
    redis_host: str = "localhost"
    redis_ssl_port: int = 6380

settings = Settings()

# ══════════════════════════════════════════════════════════════════════════════
# Azure OpenAI client (required)
# ══════════════════════════════════════════════════════════════════════════════
from openai import AzureOpenAI

if not settings.azure_openai_api_key or not settings.azure_openai_endpoint:
    st.error(
        "❌ AZURE_OPENAI_API_KEY and AZURE_OPENAI_ENDPOINT must be set in .env file. "
        "Please add them and restart the app."
    )
    st.stop()

openai_client = AzureOpenAI(
    azure_endpoint=settings.azure_openai_endpoint,
    api_key=settings.azure_openai_api_key,
    api_version=settings.azure_openai_api_version,
)

# ══════════════════════════════════════════════════════════════════════════════
# Local clients
# ══════════════════════════════════════════════════════════════════════════════
class LocalRedis:
    def __init__(self):
        self._store: dict[str, str] = {}
    def get(self, key: str) -> Optional[str]:
        return self._store.get(key)
    def setex(self, key: str, ttl: int, value: str):
        self._store[key] = value
    def ping(self) -> bool:
        return True

redis_client = LocalRedis()

DB_PATH = PROJECT_ROOT / "data" / "demo_local.db"
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

def _get_db() -> sqlite3.Connection:
    conn = sqlite3.connect(str(DB_PATH), check_same_thread=False)
    conn.row_factory = sqlite3.Row
    return conn

LOCAL_DOCS_DIR = PROJECT_ROOT / "data" / "raw-docs"
LOCAL_DOCS_DIR.mkdir(parents=True, exist_ok=True)

# ══════════════════════════════════════════════════════════════════════════════
# Tools (condensed from notebook sections 4a-4d)
# ══════════════════════════════════════════════════════════════════════════════
CHUNK_SIZE, CHUNK_OVERLAP = 1000, 100
_local_search_index: list[dict[str, Any]] = []

def _chunk_text(text: str) -> list[str]:
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + CHUNK_SIZE])
        start += CHUNK_SIZE - CHUNK_OVERLAP
    return chunks

def local_ingest_directory(directory: Path) -> dict[str, int]:
    global _local_search_index
    files = sorted([*directory.glob("*.md"), *directory.glob("*.txt")])
    count = 0
    for fp in files:
        text = fp.read_text(encoding="utf-8", errors="ignore")
        for i, chunk in enumerate(_chunk_text(text)):
            doc_id = hashlib.md5(f"{fp.name}:{i}".encode()).hexdigest()
            _local_search_index.append({"id": doc_id, "content": chunk, "title": fp.name, "source": str(fp)})
            count += 1
    return {"files_indexed": len(files), "chunks_indexed": count}

def hybrid_search(query: str, top_k: int = 5) -> list[dict[str, Any]]:
    qw = set(query.lower().split())
    scored = [({**d, "score": sum(1 for w in qw if w in d["content"].lower())}) for d in _local_search_index]
    return sorted([s for s in scored if s["score"] > 0], key=lambda x: x["score"], reverse=True)[:top_k]

def retrieve_context(query: str, top_k: int = 5) -> tuple[str, list[str]]:
    docs = hybrid_search(query, top_k)
    parts, srcs, total = [], [], 0
    for d in docs:
        chunk = f"[{d['title']}]\n{d['content']}"
        if total + len(chunk) > 6000:
            break
        parts.append(chunk); srcs.append(d["source"]); total += len(chunk)
    return "\n\n---\n\n".join(parts), list(set(srcs))

def query_incident_history(service_name: str, limit: int = 10) -> list[dict]:
    with _get_db() as conn:
        return [dict(r) for r in conn.execute(
            "SELECT * FROM incidents WHERE service = ? ORDER BY started_at DESC LIMIT ?",
            (service_name, limit)).fetchall()]

def query_service_dependencies(service_name: str) -> list[dict]:
    with _get_db() as conn:
        return [dict(r) for r in conn.execute(
            "SELECT upstream, downstream, dependency_type FROM service_dependencies WHERE upstream = ? OR downstream = ?",
            (service_name, service_name)).fetchall()]

def log_agent_interaction(session_id: str, agent: str, query: str, answer: str):
    with _get_db() as conn:
        conn.execute("INSERT INTO agent_interactions (session_id, agent, query, answer) VALUES (?, ?, ?, ?)",
                     (session_id, agent, query, answer))

def get_conversation_history(session_id: str) -> list[dict]:
    raw = redis_client.get(f"session:{session_id}:history")
    return json.loads(raw) if raw else []

def append_to_history(session_id: str, role: str, content: str):
    h = get_conversation_history(session_id)
    h.append({"role": role, "content": content})
    redis_client.setex(f"session:{session_id}:history", 3600, json.dumps(h[-20:]))

import builtins as _bm
_AB = {"print","len","range","enumerate","zip","list","dict","set","tuple","str","int","float","bool","type","isinstance","min","max","sum"}
def execute_code(code: str) -> dict[str, Any]:
    rg = {"__builtins__": {k: getattr(_bm, k) for k in _AB if hasattr(_bm, k)}}
    ol: list[str] = []
    rg["__builtins__"]["print"] = lambda *a, **kw: ol.append(" ".join(str(x) for x in a))
    try:
        lv: dict = {}
        exec(compile(code, "<sandbox>", "exec"), rg, lv)
        return {"status": "ok", "output": "\n".join(ol)}
    except Exception:
        return {"status": "error", "output": traceback.format_exc(limit=5)}

_STOP_WORDS = {"what","when","where","which","there","their","about","would","could","should","have","been","that","this","with","from","your","into","will","more","also"}

# ══════════════════════════════════════════════════════════════════════════════
# Agent runners
# ══════════════════════════════════════════════════════════════════════════════
SRE_PROMPT = "You are an expert SRE AI assistant. Analyze incidents, suggest RCA and remediation. Ground answers in retrieved context. Be concise."
ENG_PROMPT = "You are an expert Software Engineering AI assistant. Answer architecture and code questions. Ground answers in retrieved docs. Be precise."

def _run_agent(session_id: str, user_message: str, agent_type: str) -> dict[str, Any]:
    prefix = "sre" if agent_type == "sre" else "eng"
    system_prompt = SRE_PROMPT if agent_type == "sre" else ENG_PROMPT
    history = get_conversation_history(f"{prefix}:{session_id}")
    context, sources = retrieve_context(user_message, top_k=5)
    tool_calls: list[str] = []

    # DB lookup
    db_context = ""
    for word in user_message.lower().split():
        if len(word) > 4 and word not in _STOP_WORDS:
            if agent_type == "sre":
                rows = query_incident_history(word)
                if rows:
                    db_context = "Recent incidents:\n" + "\n".join(f"- [{r['severity']}] {r['title']}: {r.get('summary','')}" for r in rows)
                    tool_calls.append(f"query_incident_history(service={word})")
                    break
            else:
                rows = query_service_dependencies(word)
                if rows:
                    db_context = "Service dependencies:\n" + "\n".join(f"- {r['upstream']} → {r['downstream']} ({r['dependency_type']})" for r in rows)
                    tool_calls.append(f"query_service_dependencies(service={word})")
                    break

    sandbox_result = None
    if "```python" in user_message:
        s = user_message.find("```python") + 9
        e = user_message.find("```", s)
        if e > s:
            sandbox_result = execute_code(user_message[s:e].strip())
            tool_calls.append("execute_code(sandbox)")

    msgs = [{"role": "system", "content": system_prompt}]
    if context:
        msgs.append({"role": "system", "content": f"Retrieved context:\n\n{context}"})
    if db_context:
        msgs.append({"role": "system", "content": db_context})
    if sandbox_result:
        msgs.append({"role": "system", "content": f"Sandbox result:\n{sandbox_result['output']}"})
    msgs.extend(history[-10:])
    msgs.append({"role": "user", "content": user_message})

    # Call Azure OpenAI
    try:
        resp = openai_client.chat.completions.create(
            model=settings.azure_openai_deployment,
            messages=msgs,
            temperature=0.2,
            max_tokens=1500,
        )
        answer = resp.choices[0].message.content
    except Exception as exc:
        answer = f"[LLM error: {exc}]\n\nContext:\n{context[:500]}" if context else f"[LLM error: {exc}]"

    append_to_history(f"{prefix}:{session_id}", "user", user_message)
    append_to_history(f"{prefix}:{session_id}", "assistant", answer)
    try:
        log_agent_interaction(session_id, agent_type, user_message, answer)
    except Exception:
        pass
    return {"answer": answer, "sources": sources, "tool_calls": tool_calls}

# ══════════════════════════════════════════════════════════════════════════════
# Startup: ingest local docs
# ══════════════════════════════════════════════════════════════════════════════
local_ingest_directory(LOCAL_DOCS_DIR)

# ══════════════════════════════════════════════════════════════════════════════
# Streamlit UI
# ══════════════════════════════════════════════════════════════════════════════
st.set_page_config(page_title="Agentic RAG Chat", page_icon="🤖", layout="wide")
st.title("🤖 Agentic RAG Platform — Local Demo")

tab_chat, tab_ingest, tab_health = st.tabs(["💬 Chat", "📄 Ingest Documents", "🩺 Health"])

# ── Chat Tab ─────────────────────────────────────────────────────────────────
with tab_chat:
    col1, col2 = st.columns([1, 3])
    with col1:
        agent_type = st.radio("Agent", ["sre", "engineering"], index=0)
        session_id = st.text_input("Session ID", value=str(uuid.uuid4())[:8])

    with col2:
        if "messages" not in st.session_state:
            st.session_state.messages = []

        for msg in st.session_state.messages:
            with st.chat_message(msg["role"]):
                st.markdown(msg["content"])

        if prompt := st.chat_input("Ask the agent..."):
            st.session_state.messages.append({"role": "user", "content": prompt})
            with st.chat_message("user"):
                st.markdown(prompt)

            with st.chat_message("assistant"):
                with st.spinner("Thinking..."):
                    result = _run_agent(session_id, prompt, agent_type)
                st.markdown(result["answer"])
                if result["sources"]:
                    st.caption(f"📚 Sources: {', '.join(result['sources'])}")
                if result["tool_calls"]:
                    st.caption(f"🔧 Tools used: {', '.join(result['tool_calls'])}")

            st.session_state.messages.append({"role": "assistant", "content": result["answer"]})

# ── Ingest Tab ───────────────────────────────────────────────────────────────
with tab_ingest:
    st.subheader("Upload documents to the knowledge base")
    uploaded = st.file_uploader("Choose .md or .txt files", type=["md", "txt"], accept_multiple_files=True)
    if uploaded and st.button("Ingest"):
        for f in uploaded:
            dest = LOCAL_DOCS_DIR / f.name
            dest.write_bytes(f.getvalue())
        stats = local_ingest_directory(LOCAL_DOCS_DIR)
        st.success(f"Ingested {stats['chunks_indexed']} chunks from {stats['files_indexed']} file(s)")

# ── Health Tab ───────────────────────────────────────────────────────────────
with tab_health:
    st.subheader("Service Health")
    svc = {}
    svc["Redis"] = "✅ ok (local in-memory)"
    try:
        with _get_db() as c:
            c.execute("SELECT 1")
        svc["PostgreSQL"] = "✅ ok (local SQLite)"
    except Exception as e:
        svc["PostgreSQL"] = f"❌ {e}"
    svc["Azure OpenAI"] = "✅ configured (API key auth)"
    svc["Search Index"] = f"✅ {len(_local_search_index)} chunks indexed"
    for name, status in svc.items():
        st.markdown(f"**{name}**: {status}")
'''

# Write the Streamlit app to disk
streamlit_path = PROJECT_ROOT / "notebooks" / "streamlit_app.py"
streamlit_path.write_text(streamlit_app_code.strip(), encoding="utf-8")
print(f"✅ Streamlit app written to: {streamlit_path}")
print()
print("To run the Streamlit app, open a terminal and execute:")
print(f"    cd {PROJECT_ROOT / 'notebooks'}")
print("    streamlit run streamlit_app.py")

✅ Streamlit app written to: c:\Project\AI\rag-infra\notebooks\streamlit_app.py

To run the Streamlit app, open a terminal and execute:
    cd c:\Project\AI\rag-infra\notebooks
    streamlit run streamlit_app.py


## 10. Auth Flow (Reference)

In the production app, authentication is handled by `app/core/auth.py` using **Azure Entra ID (AAD)** JWT tokens.

The Streamlit demo does **not** include authentication. Below is the reference implementation for context.

> To add auth to Streamlit, you could use [streamlit-authenticator](https://github.com/mkhorasani/Streamlit-Authenticator) or Azure AD MSAL.

In [ ]:
# ── Auth reference (from app/core/auth.py) ────────────────────────────────────
# This section is informational only — auth is NOT enforced in the local demo.
#
# Production auth flow:
#   1. Client sends a Bearer token in the Authorization header.
#   2. The token is a JWT issued by Azure Entra ID.
#   3. The server validates the JWT against the JWKS endpoint:
#        https://login.microsoftonline.com/{tenant_id}/discovery/v2.0/keys
#   4. The JWT audience must match settings.entra_audience.
#   5. Role-based access (require_role) checks the "roles" claim.
#
# To add auth to the Streamlit app:
#   - Use streamlit-authenticator for simple user/password auth.
#   - Or use msal + streamlit to do Azure AD login flow.
#   - Or protect the Streamlit app behind Azure App Service Authentication.
#
# Example production guard:
#
# async def get_current_user(creds) -> dict:
#     token = creds.credentials
#     jwks = await _get_jwks()
#     payload = jwt.decode(token, jwks, algorithms=["RS256"], audience=settings.entra_audience)
#     return payload
#
# def require_role(*roles):
#     async def _guard(user=Depends(get_current_user)):
#         if not any(r in user.get("roles", []) for r in roles):
#             raise HTTPException(403, "Insufficient role")
#         return user
#     return _guard

print("ℹ️  Auth section is reference-only — no authentication is enforced in this local demo.")

## 11. Azure Blob Storage Integration (Reference)

In production, document ingestion reads from **Azure Blob Storage** (`app/services/ingestion_service.py`) and the documents listing comes from `app/api/routes/documents.py`.

Locally this is replaced by the `data/raw-docs/` directory. Below is the reference for how the production blob integration works.

In [ ]:
# ── Blob Storage reference (from app/services/ingestion_service.py) ────────────
# This section is informational only — locally we use data/raw-docs/ directory.
#
# Production ingestion flow:
#   1. Client calls POST /ingest with container name and optional blob prefix.
#   2. The service iterates over blobs in the Azure Blob Storage container.
#   3. Each .txt / .md / .pdf file is downloaded and chunked (1000 chars, 100 overlap).
#   4. Each chunk is embedded via Azure OpenAI (text-embedding-3-large).
#   5. Chunks are uploaded to the Azure AI Search index with vector + metadata.
#
# Production document listing flow:
#   1. Client calls GET /documents?container=raw-docs
#   2. The route lists all blobs in the container via BlobServiceClient.
#   3. Returns name, size, last_modified, and URI for each blob.
#
# Local equivalent:
#   - Place .md or .txt files in data/raw-docs/
#   - Run ingest_local_docs() or use the Streamlit "Ingest Documents" tab
#   - Documents are chunked and stored in the in-memory _local_search_index

# List locally available documents
local_docs = sorted(LOCAL_DOCS_DIR.glob("*"))
print(f"📁 Documents in {LOCAL_DOCS_DIR}:")
for doc in local_docs:
    size = doc.stat().st_size if doc.is_file() else 0
    print(f"  {doc.name:40s}  {size:>8,d} bytes")

## 12. Azure AI Search Index Creation (Reference)

In production, the RAG service (`app/services/rag_service.py`) auto-creates the Azure AI Search index with vector search and semantic configuration.

This is not needed locally since we use a simple in-memory index. Below is the reference schema.

In [ ]:
# ── Azure AI Search index reference (from app/services/rag_service.py) ────────
# This section is informational only — locally we use an in-memory list.
#
# Production search index schema:
#   Fields:
#     - id           : String (key)
#     - content      : Searchable String
#     - title        : Searchable String (filterable)
#     - source       : Simple String (filterable)
#     - content_vector: Collection(Single), 3072 dimensions, HNSW profile
#
#   Vector search:
#     - Algorithm: HNSW (hnsw-algo)
#     - Profile: hnsw-profile
#
#   Semantic search:
#     - Configuration: semantic-config
#     - Content field: content
#     - Title field: title
#
#   Hybrid search (keyword + vector):
#     - query_text  → keyword search on content + title
#     - query_vector → HNSW nearest-neighbor search on content_vector
#     - Results merged by reciprocal rank fusion
#
# The ensure_index_exists() function in rag_service.py creates this index
# automatically on first use if it doesn't already exist.

print("ℹ️  AI Search index section is reference-only — local demo uses in-memory keyword search.")

## 13. Audit Log — Review Agent Interactions

All agent interactions are persisted in the local SQLite database. This is useful for debugging and reviewing agent behaviour.

In [ ]:
with _get_db() as conn:
    rows = conn.execute(
        "SELECT session_id, agent, query, created_at FROM agent_interactions ORDER BY created_at DESC LIMIT 10"
    ).fetchall()

print(f"Recent agent interactions ({len(rows)}):")
print("-" * 90)
for r in rows:
    row = dict(r)
    print(f"  [{row['created_at']}] {row['agent']:12s} session={row['session_id']}")
    print(f"    Q: {row['query'][:80]}")
    print()

---

## Summary

This notebook demonstrated every layer of the **Agentic RAG Platform** running locally:

| Section | Production Component | Local Substitute |
|---|---|---|
| Settings | `app/core/config.py` | Same Pydantic Settings (reads `.env`) |
| Schemas | `app/models/schemas.py` | Same Pydantic models |
| Clients | Azure MI + Key Vault + Blob + PG + Redis | Azure OpenAI (API key) + SQLite + dict + local files |
| Search Tool | Azure AI Search (hybrid) | In-memory keyword search |
| Postgres Tool | Azure PostgreSQL | Local SQLite |
| Redis Tool | Azure Cache for Redis | In-memory dict |
| Sandbox Tool | Restricted `exec()` | Same |
| RAG Service | Azure AI Search + Azure OpenAI Embeddings | Local keyword search |
| SRE Agent | `app/agents/sre_agent.py` | Synchronous local version |
| Engineering Agent | `app/agents/engineering_agent.py` | Synchronous local version |
| API / UI | FastAPI + Entra ID auth | **Streamlit** (no auth) |
| Health Check | `app/api/routes/health.py` | Local health function |
| Ingestion | Azure Blob → AI Search | Local files → in-memory index |

### Next Steps
- Add `.md` / `.txt` runbooks to `data/raw-docs/` to enrich the knowledge base
- Set `AZURE_OPENAI_API_KEY` and `AZURE_OPENAI_ENDPOINT` in `.env` for real LLM responses
- Run `streamlit run notebooks/streamlit_app.py` for the interactive chat UI